# Week 1 · Day 1 — Environment Setup & Repo Scaffold

---

## 🎯 Day 1 Objective

Stand up the **complete development environment** so every future day starts from a clean, reproducible base.

| Step | Action | Output |
|------|--------|--------|
| 1 | Verify Python 3.11 + conda env | `geo-mro` env active |
| 2 | Scaffold full project directory | 20+ folders + `.gitignore` |
| 3 | `git init` + remote + branches | `main` + `develop` + CI rules |
| 4 | Install all Week 1–12 deps | `environment.yml` locked |
| 5 | Smoke-test all critical imports | Zero import errors |

---

## ❓ Why This Matters

Every downstream module (ABC classifier, Croston's engine, Newsvendor, GeoRisk) imports from `src/`.
If the repo structure or conda env is wrong → **everything breaks**.
Day 1 is the foundation. Do it once, do it right.

---

## ⚙️ Architecture (end-state of Day 1)

```
geo-aware-mro/
├─ data/          raw/ · processed/ · external/
├─ notebooks/     01_sku_intelligence/ · 02_demand_forecasting/ · ...
├─ src/           classifiers/ · forecasting/ · risk/ · game_theory/ · utils/ · config/
├─ tests/         unit/ · integration/
├─ mlflow/        mlruns/ (gitignored)
├─ docs/          mkdocs scaffold
├─ .github/       workflows/ci.yml
├─ environment.yml
├─ Dockerfile     (stub — built fully on Day 4)
└─ README.md      v0.1
```

---
## STEP 0 — Verify Python version

In [1]:
import sys
import platform

print(f"Python  : {sys.version}")
print(f"Platform: {platform.system()} {platform.release()}")
print(f"Path    : {sys.executable}")

assert sys.version_info >= (3, 11), (
    f"❌ Need Python ≥ 3.11, got {sys.version_info.major}.{sys.version_info.minor}. "
    "Run: conda activate geo-mro"
)
print("\n✅ Python 3.11+ confirmed")

Python  : 3.14.4 (tags/v3.14.4:23116f9, Apr  7 2026, 14:10:54) [MSC v.1944 64 bit (AMD64)]
Platform: Windows 11
Path    : c:\Users\DELL\AppData\Local\Programs\Python\Python314\python.exe

✅ Python 3.11+ confirmed


---
## STEP 1 — Project root & path setup

In [2]:
from pathlib import Path
import sys

# ── Resolve project root robustly (works from notebooks/ OR repo root) ──
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    # Running inside Jupyter — walk up until we find pyproject.toml or .git
    cwd = Path.cwd()
    ROOT = cwd
    for parent in [cwd] + list(cwd.parents):
        if (parent / ".git").exists() or (parent / "pyproject.toml").exists():
            ROOT = parent
            break

# Add src/ to Python path so `from src.xxx import yyy` works everywhere
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"📁 Project root : {ROOT}")
print(f"📦 src/ on path : {SRC.exists()}")

📁 Project root : d:\geo-aware-mro
📦 src/ on path : False


---
## STEP 2 — Scaffold full project directory

In [3]:
from pathlib import Path

# ── Full directory tree (matches v1.1 end-state from roadmap slide 9) ──
DIRS: list[str] = [
    # Data layers
    "data/raw",
    "data/processed",
    "data/external",
    "data/interim",           # staging between raw → processed

    # Notebooks — one folder per module
    "notebooks/01_sku_intelligence",
    "notebooks/02_demand_forecasting",
    "notebooks/03_risk_scoring",
    "notebooks/04_supplier_qualification",
    "notebooks/05_exploratory",

    # Source packages
    "src/classifiers",
    "src/forecasting",
    "src/risk",
    "src/game_theory",
    "src/utils",
    "src/config",
    "src/dashboard",

    # Tests
    "tests/unit",
    "tests/integration",

    # MLflow
    "mlflow/mlruns",

    # Docs
    "docs/api",
    "docs/guides",

    # CI
    ".github/workflows",
]

created, existed = [], []

for d in DIRS:
    full = ROOT / d
    if not full.exists():
        full.mkdir(parents=True, exist_ok=True)
        # Python packages need __init__.py
        if d.startswith("src/") or d.startswith("tests/"):
            (full / "__init__.py").touch()
        # Keep empty dirs in git
        else:
            (full / ".gitkeep").touch()
        created.append(d)
    else:
        existed.append(d)

# Top-level src __init__.py
(ROOT / "src" / "__init__.py").touch(exist_ok=True)
(ROOT / "tests" / "__init__.py").touch(exist_ok=True)

print(f"✅ Created  : {len(created)} dirs")
print(f"⏩ Already  : {len(existed)} dirs")
for d in created:
    print(f"   + {d}")

✅ Created  : 22 dirs
⏩ Already  : 0 dirs
   + data/raw
   + data/processed
   + data/external
   + data/interim
   + notebooks/01_sku_intelligence
   + notebooks/02_demand_forecasting
   + notebooks/03_risk_scoring
   + notebooks/04_supplier_qualification
   + notebooks/05_exploratory
   + src/classifiers
   + src/forecasting
   + src/risk
   + src/game_theory
   + src/utils
   + src/config
   + src/dashboard
   + tests/unit
   + tests/integration
   + mlflow/mlruns
   + docs/api
   + docs/guides
   + .github/workflows


---
## STEP 3 — Write environment.yml

This is the single source of truth for all 12-week dependencies.
Pinned to minor versions → reproducible across machines and Docker.

In [4]:
ENVIRONMENT_YML = """\
name: geo-mro
channels:
  - conda-forge
  - defaults

dependencies:
  # ── Core runtime ──────────────────────────────────────────────────────
  - python=3.11
  - pip

  # ── Data science stack ────────────────────────────────────────────────
  - numpy>=1.26
  - pandas>=2.1
  - scipy>=1.11
  - scikit-learn>=1.3
  - statsmodels>=0.14

  # ── Forecasting ───────────────────────────────────────────────────────
  - pip:
    - sktime>=0.26           # Croston, SBA, unified forecasting API
    - pmdarima>=2.0          # auto-ARIMA

  # ── MLOps ─────────────────────────────────────────────────────────────
  - mlflow>=2.9
  - pip:
    - dvc>=3.30              # data versioning
    - dvc-s3                 # optional S3 remote

  # ── Data engineering ─────────────────────────────────────────────────
  - duckdb>=0.9
  - pyarrow>=14.0
  - fastparquet>=2023.8
  - requests>=2.31
  - pip:
    - faker>=20.0            # synthetic SKU master generation

  # ── Dashboard ─────────────────────────────────────────────────────────
  - plotly>=5.18
  - pip:
    - dash>=2.14
    - dash-bootstrap-components>=1.5

  # ── Game theory / simulation ──────────────────────────────────────────
  - pip:
    - nashpy>=0.0.19         # Nash equilibrium (W10)
    - simpy>=4.0             # discrete-event simulation

  # ── Parallel / performance ────────────────────────────────────────────
  - joblib>=1.3

  # ── Dev tooling ───────────────────────────────────────────────────────
  - jupyter>=1.0
  - ipykernel>=6.27
  - black>=23.12
  - ruff>=0.1
  - pytest>=7.4
  - pytest-cov>=4.1
  - pip:
    - mkdocs>=1.5            # documentation site
    - mkdocs-material>=9.4
"""

env_path = ROOT / "environment.yml"
env_path.write_text(ENVIRONMENT_YML)
print(f"✅ Written: {env_path}")

UnicodeEncodeError: 'charmap' codec can't encode characters in position 78-79: character maps to <undefined>

---
## STEP 4 — Write src/config/settings.py

Central settings object used by ALL notebooks and modules.
One import → all paths resolved.

In [ ]:
SETTINGS_PY = '''\
# src/config/settings.py
# Central configuration — import this everywhere, never hardcode paths.

from __future__ import annotations
from pathlib import Path
from dataclasses import dataclass, field


@dataclass(frozen=True)
class Settings:
    # ── Project root ──────────────────────────────────────────────────────
    ROOT: Path = field(
        default_factory=lambda: Path(__file__).resolve().parents[2]
    )

    # ── Data paths ────────────────────────────────────────────────────────
    @property
    def DATA_DIR(self) -> Path:      return self.ROOT / "data"
    @property
    def RAW_DIR(self) -> Path:       return self.ROOT / "data" / "raw"
    @property
    def PROCESSED_DIR(self) -> Path: return self.ROOT / "data" / "processed"
    @property
    def EXTERNAL_DIR(self) -> Path:  return self.ROOT / "data" / "external"

    # ── MLflow ────────────────────────────────────────────────────────────
    @property
    def MLFLOW_DIR(self) -> Path:    return self.ROOT / "mlflow" / "mlruns"
    @property
    def MLFLOW_URI(self) -> str:     return f"sqlite:///{self.MLFLOW_DIR / \"mlflow.db\"}"

    # ── Project constants ─────────────────────────────────────────────────
    PROJECT_NAME: str = "geo-aware-mro"
    VERSION:      str = "0.1.0"
    N_SKUS:       int = 500          # synthetic SKU master size (W2)
    RANDOM_SEED:  int = 42

    # ── Ci weights (tuned in W4) ──────────────────────────────────────────
    W_ABC:  float = 0.35
    W_VED:  float = 0.30
    W_FNS:  float = 0.20
    W_LOC:  float = 0.15

    def __post_init__(self):
        # Ensure critical dirs exist at import time
        for d in [self.RAW_DIR, self.PROCESSED_DIR, self.EXTERNAL_DIR, self.MLFLOW_DIR]:
            d.mkdir(parents=True, exist_ok=True)


# ── Singleton — import this object, don\'t instantiate ───────────────────────
settings = Settings()
'''

config_dir = ROOT / "src" / "config"
config_dir.mkdir(parents=True, exist_ok=True)
(config_dir / "__init__.py").touch(exist_ok=True)
(config_dir / "settings.py").write_text(SETTINGS_PY)
print(f"✅ Written: src/config/settings.py")

---
## STEP 5 — Write src/utils/logger.py

In [ ]:
LOGGER_PY = '''\
# src/utils/logger.py
# Shared logger — `from src.utils.logger import get_logger`

import logging
import sys

_FMT = "%(asctime)s | %(levelname)s | %(name)s | %(message)s"
_DATE = "%Y-%m-%d %H:%M:%S"


def get_logger(name: str, level: int = logging.INFO) -> logging.Logger:
    logger = logging.getLogger(name)
    if logger.handlers:          # avoid duplicate handlers on re-import
        return logger
    handler = logging.StreamHandler(sys.stderr)
    handler.setFormatter(logging.Formatter(_FMT, datefmt=_DATE))
    logger.addHandler(handler)
    logger.setLevel(level)
    logger.propagate = False
    return logger
'''

utils_dir = ROOT / "src" / "utils"
utils_dir.mkdir(parents=True, exist_ok=True)
(utils_dir / "__init__.py").touch(exist_ok=True)
(utils_dir / "logger.py").write_text(LOGGER_PY)
print(f"✅ Written: src/utils/logger.py")

---
## STEP 6 — Write .gitignore

In [ ]:
GITIGNORE = """\
# Python
__pycache__/
*.py[cod]
*.egg-info/
.eggs/
dist/
build/

# Jupyter
.ipynb_checkpoints/

# Environments
.env
.venv/
env/

# Data — tracked by DVC, not git
data/raw/*
data/processed/*
data/external/*
data/interim/*
!data/**/.gitkeep
!data/**/*.dvc

# MLflow — runs stored locally, not committed
mlflow/mlruns/
*.db

# IDE
.vscode/
.idea/
*.code-workspace

# OS
.DS_Store
Thumbs.db

# Secrets
.env.local
secrets/
*.pem
*.key

# Docs build
site/
"""

(ROOT / ".gitignore").write_text(GITIGNORE)
print("✅ Written: .gitignore")

---
## STEP 7 — Write README v0.1

In [ ]:
README = """\
# Geo-Aware MRO Decision Intelligence System

![version](https://img.shields.io/badge/version-0.1.0-blue)
![python](https://img.shields.io/badge/python-3.11-blue)
![license](https://img.shields.io/badge/license-MIT-green)

> **Status:** Week 1 — Infrastructure scaffold in progress.

---

## Problem Statement

MRO (Maintenance, Repair & Overhaul) inventory decisions in industrial supply chains are
driven by three compounding uncertainties:

1. **Demand** is intermittent, lumpy, and highly SKU-specific.
2. **Supply** is geographically concentrated — a geopolitical shock in one country
   can disable an entire production line.
3. **Criticality** is multi-dimensional — a $2 seal and a $40,000 turbine blade
   require fundamentally different stocking policies.

This system fuses **27-class SKU taxonomy**, **Bayesian geo-risk scoring**,
**Croston-family demand forecasting**, and **Newsvendor-optimal order quantities**
into a single, auditable decision intelligence pipeline.

---

## Architecture

```
geo-aware-mro/
├─ data/          raw/ · processed/ · external/        [DVC-tracked]
├─ notebooks/     01–04 module notebooks
├─ src/           classifiers/ · forecasting/ · risk/  [importable]
├─ tests/         pytest suite (target ≥80% coverage)
├─ mlflow/        experiment tracking
├─ docs/          mkdocs site → GitHub Pages
└─ Dockerfile     reproducible container
```

## Quickstart

```bash
# 1. Clone
git clone https://github.com/<your-username>/geo-aware-mro.git
cd geo-aware-mro

# 2. Create env
conda env create -f environment.yml
conda activate geo-mro

# 3. Pull data
dvc pull

# 4. Run tests
pytest tests/ --cov=src
```

## Roadmap

| Phase | Weeks | Focus |
|-------|-------|-------|
| 1 | W1–W4 | Infrastructure + SKU Intelligence |
| 2 | W5–W8 | Analytics Engine (Forecasting + GeoRisk) |
| 3 | W9–W12 | Strategy + v1.1 GitHub Release |

---

*Built by Deepender — Decision Science · Supply Chain Analytics · Industrial AI*
"""

(ROOT / "README.md").write_text(README)
print("✅ Written: README.md v0.1")

---
## STEP 8 — Write GitHub Actions CI pipeline

Minimal CI: lint → test → import-smoke on every push to `main` or `develop`.

In [ ]:
CI_YML = """\
name: CI

on:
  push:
    branches: [main, develop]
  pull_request:
    branches: [main, develop]

jobs:
  test:
    runs-on: ubuntu-latest
    defaults:
      run:
        shell: bash -l {0}          # login shell so conda works

    steps:
      - uses: actions/checkout@v4

      - name: Set up Conda
        uses: conda-incubator/setup-miniconda@v3
        with:
          environment-file: environment.yml
          activate-environment: geo-mro
          auto-activate-base: false

      - name: Lint (ruff)
        run: ruff check src/ tests/

      - name: Format check (black)
        run: black --check src/ tests/

      - name: Run tests
        run: pytest tests/ --cov=src --cov-report=term-missing -q

      - name: Import smoke test
        run: |
          python -c "from src.config.settings import settings; print(settings.VERSION)"
          python -c "from src.utils.logger import get_logger; get_logger('ci')"
"""

ci_path = ROOT / ".github" / "workflows" / "ci.yml"
ci_path.parent.mkdir(parents=True, exist_ok=True)
ci_path.write_text(CI_YML)
print("✅ Written: .github/workflows/ci.yml")

---
## STEP 9 — Write Dockerfile stub

Stub today — fully built on Day 4 (Thu).

In [ ]:
DOCKERFILE = """\
# ── Base: slim Python 3.11 on Debian bookworm ──────────────────────────────
FROM python:3.11-slim-bookworm

# Metadata
LABEL maintainer="Deepender"
LABEL project="geo-aware-mro"
LABEL version="0.1.0-stub"

# NOTE: This is a DAY 1 stub.
#       Full multi-stage Dockerfile built on Day 4 (Thu W1).
#       Do NOT use in production yet.

WORKDIR /app

# Install system deps
RUN apt-get update && apt-get install -y --no-install-recommends \\
    git curl build-essential \\
    && rm -rf /var/lib/apt/lists/*

# Copy and install Python deps first (Docker layer caching)
COPY environment.yml .
# TODO Day 4: convert to pip requirements for lighter image

COPY . .

# Placeholder entrypoint — replaced Day 4
CMD ["python", "-c", "from src.config.settings import settings; print('geo-mro OK')"]\n"""

(ROOT / "Dockerfile").write_text(DOCKERFILE)
print("✅ Written: Dockerfile (stub)")

---
## STEP 10 — mkdocs scaffold

In [ ]:
MKDOCS_YML = """\
site_name: Geo-Aware MRO Decision Intelligence
site_author: Deepender
site_description: 27-class SKU taxonomy · Bayesian geo-risk · Newsvendor optimization

repo_url: https://github.com/<your-username>/geo-aware-mro

theme:
  name: material
  palette:
    primary: indigo
    accent: blue

nav:
  - Home: index.md
  - Architecture: guides/architecture.md
  - API Reference: api/index.md

plugins:
  - search
"""

DOCS_INDEX = """\
# Geo-Aware MRO Decision Intelligence System

Welcome to the documentation site.

This system fuses **27-class SKU taxonomy**, **Bayesian geo-risk scoring**,
and **Newsvendor-optimal order quantities** into a single decision intelligence pipeline.

## Quick links

- [Architecture](guides/architecture.md)
- [API Reference](api/index.md)
"""

(ROOT / "mkdocs.yml").write_text(MKDOCS_YML)
(ROOT / "docs" / "index.md").write_text(DOCS_INDEX)
(ROOT / "docs" / "guides").mkdir(exist_ok=True)
(ROOT / "docs" / "guides" / "architecture.md").write_text("# Architecture\n\nTODO: fill in W4.\n")
(ROOT / "docs" / "api").mkdir(exist_ok=True)
(ROOT / "docs" / "api" / "index.md").write_text("# API Reference\n\nTODO: auto-generated from docstrings W12.\n")
print("✅ Written: mkdocs.yml + docs/ scaffold")

---
## STEP 11 — Git init + first commit

Run these **in your terminal** (not inside Jupyter).
The cell below prints the exact commands — copy and run them.

In [ ]:
GIT_COMMANDS = f"""
# ── Run these in your terminal from: {ROOT} ──────────────────────────────

cd "{ROOT}"

# 1. Init repo (skip if already done)
git init
git config user.name  "Deepender"
git config user.email "your-email@example.com"   # ← update this

# 2. First commit on main
git add .
git commit -m "chore: W1D1 — project scaffold, conda env, CI pipeline"

# 3. Create develop branch
git checkout -b develop
git checkout main

# 4. Push to GitHub
#    (create the repo on github.com first, then:)
git remote add origin https://github.com/<your-username>/geo-aware-mro.git
git push -u origin main
git push -u origin develop
"""

print(GIT_COMMANDS)

---
## STEP 12 — Branch protection (GitHub UI instructions)

After pushing, set branch protection rules on GitHub:

**For `main` branch:**
```
GitHub → Settings → Branches → Add rule
Branch name pattern : main
☑ Require pull request before merging
  ☑ Require 1 approving review
☑ Require status checks to pass before merging
  Search and add: "test" (your CI job name)
☑ Do not allow bypassing the above settings
```

**For `develop` branch:**
```
Same settings but CI check only — no review required
(develop is your daily working branch)
```

**OR via GitHub CLI** (if `gh` is installed):

In [ ]:
GH_CLI_COMMANDS = """
# ── Branch protection via GitHub CLI ─────────────────────────────────────────
# Install: https://cli.github.com/

REPO="<your-username>/geo-aware-mro"   # ← update

# Protect main: require PR + passing CI
gh api repos/$REPO/branches/main/protection \\
  --method PUT \\
  --field required_status_checks='{"strict":true,"contexts":["test"]}' \\
  --field enforce_admins=true \\
  --field required_pull_request_reviews='{"required_approving_review_count":1}' \\
  --field restrictions=null

# Protect develop: require passing CI only
gh api repos/$REPO/branches/develop/protection \\
  --method PUT \\
  --field required_status_checks='{"strict":true,"contexts":["test"]}' \\
  --field enforce_admins=false \\
  --field required_pull_request_reviews=null \\
  --field restrictions=null
"""
print(GH_CLI_COMMANDS)

---
## STEP 13 — Smoke test: import everything

This cell is the **Day 1 pass/fail gate**.
All imports must succeed before you close your laptop.

In [ ]:
import importlib
import traceback

CRITICAL_IMPORTS = [
    # Core data stack
    ("numpy",       "numpy"),
    ("pandas",      "pandas"),
    ("scipy",       "scipy"),
    ("sklearn",     "scikit-learn"),
    ("statsmodels", "statsmodels"),

    # MLOps
    ("mlflow",      "mlflow"),
    ("dvc",         "dvc"),

    # Data engineering
    ("duckdb",      "duckdb"),
    ("pyarrow",     "pyarrow"),
    ("requests",    "requests"),
    ("faker",       "faker"),

    # Dashboard
    ("plotly",      "plotly"),
    ("dash",        "dash"),

    # Forecasting
    ("sktime",      "sktime"),
    ("pmdarima",    "pmdarima"),

    # Parallel
    ("joblib",      "joblib"),
]

OPTIONAL_IMPORTS = [
    ("nashpy",      "nashpy"),
    ("simpy",       "simpy"),
]

passed, failed, optional_missing = [], [], []

for module, pkg in CRITICAL_IMPORTS:
    try:
        m = importlib.import_module(module)
        ver = getattr(m, "__version__", "?")
        passed.append((pkg, ver))
    except ImportError as e:
        failed.append((pkg, str(e)))

for module, pkg in OPTIONAL_IMPORTS:
    try:
        m = importlib.import_module(module)
        ver = getattr(m, "__version__", "?")
        passed.append((pkg, ver))
    except ImportError:
        optional_missing.append(pkg)

# Also test project-local imports
try:
    from src.config.settings import settings
    from src.utils.logger import get_logger
    passed.append(("src.config.settings", settings.VERSION))
    passed.append(("src.utils.logger", "OK"))
except Exception as e:
    failed.append(("src.*", str(e)))

print("\n" + "═" * 50)
print("  SMOKE TEST RESULTS")
print("═" * 50)

print(f"\n✅ PASSED ({len(passed)}):")
for pkg, ver in passed:
    print(f"   {pkg:<30} {ver}")

if optional_missing:
    print(f"\n⚠️  OPTIONAL MISSING ({len(optional_missing)}): {optional_missing}")
    print("   Install later: pip install nashpy simpy")

if failed:
    print(f"\n❌ FAILED ({len(failed)}):")
    for pkg, err in failed:
        print(f"   {pkg}: {err}")
    print("\nFix: conda env update -f environment.yml --prune")
    raise RuntimeError(f"{len(failed)} critical import(s) failed — fix before Day 2")
else:
    print("\n" + "═" * 50)
    print("  ✅ ALL CRITICAL IMPORTS PASSED — Day 1 complete")
    print("═" * 50)

---
## STEP 14 — Write Day 1 summary to MLflow

Even though the MLflow tracking server is the Day 2 task,
we log a minimal 'scaffold' run today so Day 2 has something to build on.

In [ ]:
import mlflow
from src.config.settings import settings

mlflow.set_tracking_uri(settings.MLFLOW_URI)
mlflow.set_experiment("W1_Infrastructure")

with mlflow.start_run(run_name="W1D1_scaffold") as run:
    mlflow.log_param("python_version", f"{sys.version_info.major}.{sys.version_info.minor}")
    mlflow.log_param("project_version", settings.VERSION)
    mlflow.log_param("n_skus_target",    settings.N_SKUS)
    mlflow.log_metric("dirs_created",    len(DIRS))
    mlflow.log_metric("imports_passed",  len(passed))
    mlflow.log_metric("imports_failed",  len(failed))
    mlflow.set_tag("day", "W1D1")
    mlflow.set_tag("status", "scaffold_complete")

print(f"✅ MLflow run logged: {run.info.run_id}")
print(f"   Tracking URI: {settings.MLFLOW_URI}")

---
## ✅ Day 1 Checklist

| Task | Done? |
|------|-------|
| Python 3.11 verified | ☐ |
| `geo-mro` conda env active | ☐ |
| 20+ project dirs created | ☐ |
| `environment.yml` written | ☐ |
| `src/config/settings.py` written | ☐ |
| `src/utils/logger.py` written | ☐ |
| `.gitignore` written | ☐ |
| `README.md` v0.1 written | ☐ |
| `Dockerfile` stub written | ☐ |
| `.github/workflows/ci.yml` written | ☐ |
| `mkdocs.yml` scaffold written | ☐ |
| All critical imports smoke-test ✅ | ☐ |
| MLflow W1D1 run logged | ☐ |
| `git init` + first commit on `main` | ☐ |
| `develop` branch created | ☐ |
| Pushed to GitHub | ☐ |
| Branch protection set | ☐ |

---

## 🔜 Day 2 (Tue) Preview

**MLflow tracking server (SQLite), MLproject file, dummy experiment**

- `mlflow server --backend-store-uri sqlite:///mlflow.db --port 5000`
- `MLproject` YAML with conda env + entry points
- First logged experiment with params + metrics
- Verify UI at `http://localhost:5000`